# Partial integration of a matrix with Gauss-Legendre points

Structural matrices are integrals of products of matrices, $\int \boldsymbol{B}^\top \boldsymbol{D}\,\boldsymbol{B}\,\mathrm{d}x$, where $\boldsymbol{B}$ holds the approximation functions and their derivatives. When the integrand changes over the domain, e.g. a region with a different property or load, the integral is split into parts. This notebook shows that the Gauss-Legendre quadrature, mapped onto the parts of the natural coordinate $\xi \in [-1, 1]$ of the whole domain, gives the same matrix as the quadrature over the whole domain when the integrand is the same everywhere. It uses NumPy and SciPy only, not `panels`; the mapping is that of [Partial integration of a polynomial over part of the domain](test_quadrature_partial_integration_of_domain.ipynb).

In [ ]:
import numpy as np
from scipy.special import roots_legendre

np.set_printoptions(precision=8, suppress=True)

n = 6
pts, ws = roots_legendre(n)

## The integrand

The domain $[0, 10]$ has the length $a = 10$ and the natural coordinate $\xi$, with $\mathrm{d}\xi/\mathrm{d}x = 2/a$. A single approximation function, the polynomial of fourth degree in $\xi$

$$f(\xi) = 4 + 2\xi - 4\xi^2 + 2\xi^3 - 10\xi^4,$$

fills the first column of $\boldsymbol{B}$, whose three rows are $f$, its first derivative $f_{,x} = (2/a) f_{,\xi}$ and a multiple of its second derivative, $-2 (2/a)^2 f_{,\xi\xi}$; the other two columns are zero. With $\boldsymbol{D}$ a $3 \times 3$ matrix of ones, the integrand $\boldsymbol{B}^\top \boldsymbol{D}\,\boldsymbol{B}$ is a polynomial of degree 8 in $\xi$, which the rule with $n = 6$ points, exact up to degree 11, integrates exactly.

In [ ]:
def matrix(a, xi):
    f = 4 + 2*xi - 4*xi**2 + 2*xi**3 - 10*xi**4
    f_xi = 2 - 8*xi + 6*xi**2 - 40*xi**3
    f_xixi = -8 + 12*xi - 120*xi**2
    B = np.array([[f, 0, 0],
                  [(2/a)*f_xi, 0, 0],
                  [-2*(2/a)**2*f_xixi, 0, 0]])
    return B.T @ np.ones((3, 3)) @ B

## Quadrature over a part of the domain

The part $[x_1, x_2]$ is the interval $[\xi_1, \xi_2]$ of the natural coordinate; the Gauss points are mapped onto it, $\xi' = \xi_1 + (\xi_i + 1)(\xi_2 - \xi_1)/2$, and the weights are scaled by $(x_2 - x_1)/2$, such that the result is $\int_{x_1}^{x_2} \boldsymbol{B}^\top \boldsymbol{D}\,\boldsymbol{B}\,\mathrm{d}x$.

In [ ]:
def intquad(xinf, xsup, x1, x2):
    a = xsup - xinf
    xi1 = 2*(x1 - xinf)/a - 1
    xi2 = 2*(x2 - xinf)/a - 1
    out = np.zeros((3, 3))
    for xi, w in zip(pts, ws):
        out += w*matrix(a, xi1 + (xi + 1)*(xi2 - xi1)/2)*(x2 - x1)/2
    return out

## Results

The matrix integrated over the whole domain, and as the sum of the integrals over $[0, 4]$, $[4, 6]$ and $[6, 10]$:

In [ ]:
whole = intquad(0., 10., 0., 10.)
parts = intquad(0., 10., 0., 4.) + intquad(0., 10., 4., 6.) + intquad(0., 10., 6., 10.)
print('whole domain:\n', whole)
print('sum of the three parts:\n', parts)
print('relative difference of the non-zero entry (parts - whole)/whole: %.1e'
      % ((parts[0, 0] - whole[0, 0])/whole[0, 0]))

As an independent reference, the same entry integrated with a rule of 20 points over the whole domain, which is also exact for this polynomial:

In [ ]:
pts20, ws20 = roots_legendre(20)
ref = sum(w*matrix(10., xi)[0, 0] for xi, w in zip(pts20, ws20))*10./2
print('20 points: %.10f, 6 points: %.10f, sum of the parts: %.10f, relative difference %.1e'
      % (ref, whole[0, 0], parts[0, 0], (parts[0, 0] - ref)/ref))

## Discussion

The sum of the parts equals the matrix of the whole domain to the round-off: since the mapping of the Gauss points onto each part is linear, the degree of exactness of the rule is kept on every part. A domain can therefore be split where its properties or loads change, each part integrated with the points mapped onto it, without changing the functions written in the natural coordinate of the whole domain.